# 11 - where the test error comes from, and what the models drew

**GPU runtime.** Measured on 2026-09-27 (A100 80 GB): RAM used 12 GB, Part 2 took
about 5 min (U-Net 0.6, C-RADIO 3.3, DINOv3 1.2 min); High-RAM is not needed.
Nothing is retrained and no result file is changed: everything is written to
`/content/predictions` and then to Drive as ONE zip,
`sar-transfer/predictions/sar-transfer-predictions.zip`.

- **Part 1** (CPU, ~1 min): pooled test MDE of every run with and without the
  18 Sentinel-1 images of Columbia, and their share of the error. A
  description of the test set after the fact; it changes no choice.
- **Speed check** (GPU, ~4 min): the new, faster feature copy-back gives
  identical features, and how much time it saves (measure first).
- **Part 2** (GPU): the saved U-Net (4-class), the C-RADIO linear probe (row 1),
  and the C-RADIO and satellite-DINOv3 decoders (row 3, seed 0) predict the
  122 test images again. Their zone maps and fronts are saved as images, and
  each image's error is compared with the saved result, so the pictures are
  checked to be what was scored.

Then on the PC: `python scripts/view_results.py` (see notebook 10) pulls this zip too and
adds the layers `pred_<run>_zones` and `pred_<run>_front`.

In [1]:
# --- cell: mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# <<<SARTRANSFER-SYNC-BEGIN>>>
# Empty sync cell: published notebooks carry no packed code. On your own machine,
# in the repository folder, run   python sync.py   and open the notebook again;
# this cell then holds the package source and unpacks it on the runtime.
raise RuntimeError("The sync cell is empty. On your own machine, in the repository folder, "
                   "run:  python sync.py  and reopen this notebook.")
# <<<SARTRANSFER-SYNC-END>>>

In [ ]:
# --- cell: secrets, GPU, RAM, data ---
import os, shutil
from pathlib import Path
import psutil, torch
from sartransfer.env import load_env, require, model_cache_on_drive
from sartransfer.data import inventory as inv

load_env()
require("HF_TOKEN", "gated model weights")
model_cache_on_drive()          # weights downloaded once, kept on Drive
assert torch.cuda.is_available(), "switch to a GPU runtime"
g = torch.cuda.get_device_properties(0)
print(f"gpu: {g.name} | GPU memory {g.total_memory / 1e9:.0f} GB | system RAM {psutil.virtual_memory().total / 1e9:.0f} GB "
      f"| CPU cores {os.cpu_count()}")
DRIVE = Path("/content/drive/MyDrive")
RAW, RES = DRIVE / "sar-transfer" / "raw", DRIVE / "sar-transfer" / "results"
inv.unzip_caffe(inv.fetch_caffe_zip(RAW), Path("/content/caffe"))
DATA = inv.find_data_root(Path("/content/caffe"))
df = inv.scan(DATA, subdirs=("sar_images",))
OUT = Path("/content/predictions")
OUT.mkdir(parents=True, exist_ok=True)
print(len(df), "SAR images | free disk:", f"{shutil.disk_usage('/content').free/1e9:.0f} GB")

In [4]:
# --- cell: PART 1 -- share of the test error from Columbia's Sentinel-1 images (CPU, ~1 min) ---
import pandas as pd
from sartransfer.predmaps import col_s1_breakdown

bd = col_s1_breakdown(RES)
bd.to_csv(OUT / "breakdown_col_s1.csv", index=False)
pd.set_option("display.width", 250)
cols = ["set", "run", "images", "col_s1_images", "MDE_all", "MDE_without_col_s1", "MDE_col_s1_only",
        "share_error", "share_points", "no_front", "no_front_col_s1"]
print(bd[cols].round({"MDE_all": 0, "MDE_without_col_s1": 0, "MDE_col_s1_only": 0,
                      "share_error": 3, "share_points": 3}).to_string(index=False))

pooling check: 22 of 22 runs match their saved MDE within 0.5 m
  set                                             run  images  col_s1_images  MDE_all  MDE_without_col_s1  MDE_col_s1_only  share_error  share_points  no_front  no_front_col_s1
 arm1                     cradio-v4-h__caffe__tile512     122             18   2089.0              1809.0           4808.0        0.215         0.094        37                7
 arm1                  dinov3-l-photo__caffe__tile512     122             18   2199.0              2013.0           3793.0        0.181         0.105        48                6
 arm1                    dinov3-l-sat__caffe__tile512     122             18   2805.0              2482.0           5900.0        0.199         0.095        60               11
 arm1                       saratrx-v1__unit__tile512     122             18   2796.0              2470.0           6446.0        0.189         0.082        70               11
 arm1            terramind-v1-base-s1__caffe__tile5

In [5]:
# --- cell: SPEED CHECK -- faster feature copy-back: identical values, and how much faster? (~4 min) ---
# New since 2026-09-27: features are cast to float16 on the GPU before the copy to the CPU
# (half the bytes, no numpy conversion). Must print IDENTICAL for both encoders before
# Part 2 runs; Part 2 then also checks every test image against the saved results.
from sartransfer.checks import copy_back_check

speed = [copy_back_check(e, "caffe", df, DATA, token=os.environ["HF_TOKEN"], layer=l)
         for e, l in (("cradio-v4-h", 32), ("dinov3-l-sat", 21))]
assert all(r["batches_different"] == 0 for r in speed), "copy-back changed features -- stop and check"

restricted to splits ('train',): 438 of 681 images
installing ['open_clip_torch']


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

cradio-v4-h: 651.6 M parameters loaded
cradio-v4-h L32: 298 tiles | forward 10.4 ms/tile | copy-back old 9.3 -> new 1.2 ms/tile | encoder pass 41% faster | IDENTICAL on every batch
restricted to splits ('train',): 438 of 681 images


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
dinov3-l-sat L21: 298 tiles | forward 7.8 ms/tile | copy-back old 7.8 -> new 1.2 ms/tile | encoder pass 42% faster | IDENTICAL on every batch


In [6]:
# --- cell: PART 2 -- redraw the saved test predictions (GPU, ~5 min) ---
from sartransfer.predmaps import draw_predictions

chk = draw_predictions(df, DATA, RES, OUT, token=os.environ["HF_TOKEN"])


===== unet =====
  unet: 122 maps drawn | front identical on 122, zone map identical (per-image IoU) on 122 of 122 images | largest front difference 0.0 m
  0.6 min

===== cradio-v4-h layer 32: cradio_row1, cradio_row3 =====


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 26 files:   0%|          | 0/26 [00:00<?, ?it/s]

remote code verified: 26 files == reviewed copy (nvidia/C-RADIOv4-H@0057b339059c)


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

cradio-v4-h: 651.6 M parameters loaded
restricted to splits ('test',): 122 of 681 images
  50/122  0.4 min elapsed, 0.51 s/image, ETA 0.6 min
  100/122  0.5 min elapsed, 0.29 s/image, ETA 0.1 min
  122/122  0.5 min elapsed, 0.24 s/image, ETA 0.0 min
in memory: train X (0, 0) (0.0 GB fp16), 122 test images
extraction time: waiting for CPU preparation 0.0 min, encoder 0.5 min, keeping/copying 0.0 min, total 0.5 min
  RAM used 12 of 179 GB
  cradio_row1: 122 maps drawn | front identical on 122, zone map identical (per-image IoU) on 122 of 122 images | largest front difference 0.0 m
  cradio_row3: 122 maps drawn | front identical on 122, zone map identical (per-image IoU) on 122 of 122 images | largest front difference 0.0 m
  3.3 min

===== dinov3-l-sat layer 21: dinosat_row3 =====


Loading weights:   0%|          | 0/415 [00:00<?, ?it/s]

dinov3-l-sat: 303.1 M parameters loaded
restricted to splits ('test',): 122 of 681 images
  50/122  0.3 min elapsed, 0.37 s/image, ETA 0.4 min
  100/122  0.4 min elapsed, 0.21 s/image, ETA 0.1 min
  122/122  0.4 min elapsed, 0.18 s/image, ETA 0.0 min
in memory: train X (0, 0) (0.0 GB fp16), 122 test images
extraction time: waiting for CPU preparation 0.0 min, encoder 0.4 min, keeping/copying 0.0 min, total 0.4 min
  RAM used 11 of 179 GB
  dinosat_row3: 122 maps drawn | front identical on 122, zone map identical (per-image IoU) on 122 of 122 images | largest front difference 0.0 m
  1.2 min


In [7]:
# --- cell: save everything to Drive as ONE zip, for the viewer on your PC ---
zip_path = Path(shutil.make_archive("/content/sar-transfer-predictions", "zip", OUT))
dst = DRIVE / "sar-transfer" / "predictions"
dst.mkdir(parents=True, exist_ok=True)
shutil.copy2(zip_path, dst / zip_path.name)
print(f"saved {dst / zip_path.name} ({zip_path.stat().st_size / 1e9:.2f} GB)")

saved /content/drive/MyDrive/sar-transfer/predictions/sar-transfer-predictions.zip (0.03 GB)
